# 6차 파인튜닝 (v6) — 강화 게이트 현장 조각 825줄
**v5 기각 사후 재도전.** 소급 GT 오염(~10%)을 게이트 3종으로 원천 차단한 재료:
분류번호 완전일치 필수 · 저자 2위 격차 0.1 · 접두 분류쌍(005.13/005.133) 다의성 스킵 — 육안 표본 48/48 무오염.

**v6 변경:** field 학습쌍 137→**701줄**(구간 000~600·900, 700·800은 v3 재료가 담당) · field 가중 **6→3**(v5의 low 퇴화가 과대 가중 탓이라는 진단 반영).
**평가(v5와 동일):** val 3분할 close/low/field — **field가 오르고 close·low가 안 떨어져야 채택.** 로컬 A/B(골든·걷기 재채점) 통과까지가 채택 조건.
**업로드:** `synth_rec.zip` + `real_rec_data_v3.zip` + `real_rec_data_field_v6.zip` (3개) · GPU(T4) 런타임 · 셀1 후 세션 재시작


In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 GPU paddle
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r PaddleOCR/requirements.txt
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 데이터 업로드(3개) + 층화 배합 — v6: 강화 게이트 field, 가중 3
MIX = {'close': 6, 'low': 2, 'pair': 4, 'field': 3}   # <- 배합 비율 (여기만 바꿔서 재실험)

from google.colab import files
up = files.upload()   # synth_rec.zip, real_rec_data_v3.zip, real_rec_data_field_v6.zip 셋 다 선택
!unzip -oq synth_rec.zip -d PaddleOCR/train_data/
!unzip -oq real_rec_data_v3.zip -d PaddleOCR/train_data/
!unzip -oq real_rec_data_field_v6.zip -d PaddleOCR/train_data/
import io
TAB, NL = chr(9), chr(10)
def read(p): return [l.split(TAB) for l in io.open(p, encoding='utf-8').read().splitlines() if l.strip()]
merged = ['synth_rec/train/' + p + TAB + t for p, t in read('PaddleOCR/train_data/synth_rec/train/rec_gt_train.txt')]
from collections import Counter
used = Counter()
for p, t, g in read('PaddleOCR/train_data/real_rec_data_v3/meta_train.txt'):
    merged += ['real_rec_data_v3/' + p + TAB + t] * MIX.get(g, 1); used[g] += MIX.get(g, 1)
for p, t, g in read('PaddleOCR/train_data/real_rec_data_field_v6/meta_field_train.txt'):
    merged += ['real_rec_data_field_v6/' + p + TAB + t] * MIX.get(g, 1); used[g] += MIX.get(g, 1)
va = {'close': [], 'low': [], 'field': []}
for p, t, g in read('PaddleOCR/train_data/real_rec_data_v3/meta_val.txt'):
    va['close' if g == 'close' else 'low'].append('real_rec_data_v3/' + p + TAB + t)
for p, t, g in read('PaddleOCR/train_data/real_rec_data_field_v6/meta_field_val.txt'):
    va['field'].append('real_rec_data_field_v6/' + p + TAB + t)
io.open('PaddleOCR/train_data/train_v6.txt', 'w', encoding='utf-8').write(NL.join(merged) + NL)
for k, v in va.items():
    io.open('PaddleOCR/train_data/val_' + k + '.txt', 'w', encoding='utf-8').write(NL.join(v) + NL)
io.open('PaddleOCR/train_data/val_all.txt', 'w', encoding='utf-8').write(NL.join(sum(va.values(), [])) + NL)
print('train', len(merged), '줄 · 실전 배합', dict(used))
print('val close', len(va['close']), '/ low', len(va['low']), '/ field', len(va['field']))


In [ ]:
# 3) config·사전학습 모델 자동 탐색
%cd /content
import glob, os
cfgs = glob.glob('PaddleOCR/configs/rec/**/*korean*', recursive=True)
CFG = next((c for c in cfgs if 'v5' in c.lower() and 'mobile' in c.lower()), cfgs[0] if cfgs else None)
print('사용 config:', CFG)
urls = [
 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
 'https://paddleocr.bj.bcebos.com/PP-OCRv5/multilingual/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
]
for u in urls:
    if os.system(f'wget -q {u} -O pretrain.pdparams') == 0 and os.path.getsize('pretrain.pdparams') > 1e6:
        print('사전학습 확보:', u); break

In [ ]:
# 4) 학습 (20 epochs, T4 ~60-80분) — 베스트 선택은 close+low+field 합산 val 기준
%cd /content/PaddleOCR
CFG_REL = CFG.split('PaddleOCR/')[1] if CFG.startswith('PaddleOCR/') else CFG
!python tools/train.py -c {CFG_REL}   -o Global.pretrained_model=/content/pretrain      Global.epoch_num=20      Global.save_model_dir=./output/korean_lowres_v6      Global.eval_batch_step="[0,500]"      Optimizer.lr.learning_rate=0.0001      Train.sampler.first_bs=32      Train.dataset.data_dir=./train_data      Train.dataset.label_file_list=["./train_data/train_v6.txt"]      Eval.dataset.data_dir=./train_data      Eval.dataset.label_file_list=["./train_data/val_all.txt"]


In [ ]:
# 5) close/low/field 3분리 평가 — field가 오르고 close·low가 안 떨어져야 채택
%cd /content/PaddleOCR
for name in ['val_close', 'val_low', 'val_field']:
    print('=====', name, '=====')
    !python tools/eval.py -c {CFG_REL}       -o Global.pretrained_model=./output/korean_lowres_v6/best_accuracy          Eval.dataset.data_dir=./train_data          Eval.dataset.label_file_list=["./train_data/{name}.txt"] 2>/dev/null | grep -E 'acc|norm'


In [ ]:
# 6) 추론 모델로 내보내기 + 다운로드
%cd /content/PaddleOCR
!python tools/export_model.py -c {CFG_REL}   -o Global.pretrained_model=./output/korean_lowres_v6/best_accuracy      Global.save_inference_dir=./korean_lowres_v6_rec_infer
!zip -q -r /content/korean_lowres_v6_rec_infer.zip korean_lowres_v6_rec_infer
from google.colab import files
files.download('/content/korean_lowres_v6_rec_infer.zip')
print('로컬 A/B: daelim_closeup.py --rec_dir korean_lowres_v6_rec_infer (v4 대비 골든·걷기 재채점 후 채택)')
